# Universe Coverage (Slice 6A)

OwnerLens's SEC-first canonical pipeline was validated on a handful of companies. Before designing the next normalization improvement, this notebook measures how it holds up across **24 conventional U.S. operating companies** in several industries:

```
SEC fetch → canonical history → owner economics → capital efficiency → economic value
          → compounding → capital allocation → summary → coverage
```

This is **coverage discovery**. Nothing here changes SEC normalization or adds ticker overrides. Every blocker is diagnosed from the raw Company Facts using a small read-only catalog of alternative SEC concepts.

The survey was run with `uv run python scripts/survey_universe.py`, which ingests through the existing `ingest_company` into a dedicated `data/universe_6a.db`. This notebook replays it **offline** from the stored raw snapshots, so it is reproducible and makes no SEC requests.

In [1]:
import json
from pathlib import Path

from owner_lens.persistence import FilesystemRawSnapshotStore
from owner_lens.universe import (
    UNIVERSE_6A,
    UniverseReport,
    format_company_view,
    format_metric_view,
    format_pattern_view,
    survey_from_snapshot,
)
from owner_lens.universe import DiagnosisCategory as D
from owner_lens.universe import RestatementKind as R

repo_root = next((p for p in (Path('.'), Path('..')) if (p / 'pyproject.toml').exists()), Path('.'))
manifest = json.loads((repo_root / 'data' / 'universe_6a_report.json').read_text(encoding='utf-8'))
raw_store = FilesystemRawSnapshotStore(repo_root / 'data' / 'raw' / 'sec' / 'company_facts')
report = UniverseReport(tuple(survey_from_snapshot(entry, raw_store) for entry in manifest['companies']))
print(f'{len(report.companies)} companies replayed offline from stored SEC snapshots')
for member in UNIVERSE_6A:
    if member.note:
        print(f'  {member.ticker}: {member.note}')

2 companies replayed offline from stored SEC snapshots
  CAT: stress test: captive finance arm (Cat Financial)
  DE: stress test: captive finance arm (John Deere Financial)
  UNH: stress test: insurance-like balance sheet


## 1. Company view

Each row shows the six analytical layers (`OK` available, `PART` partial, `INSUF` insufficient data, `UNAV` unsupported input, `BLOCK` invalid input, `ERR` unexpected exception) and the first blocking metric with its diagnosis.

`FULL` requires every layer to be available **and** no silently stale metric.

In [2]:
print(format_company_view(report))

Ticker Overall  Owner CapEf EcVal Comp  CapAl Summ   Unsupported                        Invalid                  Stale (silent)               Primary blocker
NKE    FAILED   UNAV  UNAV  UNAV  UNAV  UNAV  UNAV   operating_income,current_debt      short_term_investments   -                            operating_income [MISSING_CONCEPT]
LULU   PARTIAL  OK    UNAV  UNAV  UNAV  UNAV  UNAV   current_debt,long_term_debt        -                        cash                         current_debt [ALTERNATIVE_CONCEPT]

FULL=0 (0%)  PARTIAL=1 (50%)  FAILED=1 (50%)  of 2


## 2. Metric view: which canonical metrics block coverage?

For each canonical metric: how many companies have it available, unsupported, invalid (ambiguous or malformed), or structurally absent; how many companies it blocks; and the dominant diagnosis.

In [3]:
print(format_metric_view(report))

Metric                    Avail Unsup Inval Absent Blocks  Dominant diagnosis   Companies affected
revenue                       2     0     0      0      0  -                    -
operating_income              1     1     0      0      1  MISSING_CONCEPT      unsupported: NKE
net_income                    2     0     0      0      0  -                    -
operating_cash_flow           2     0     0      0      0  -                    -
capital_expenditures          2     0     0      0      0  -                    -
diluted_shares                2     0     0      0      0  -                    -
income_tax_expense            2     0     0      0      0  -                    -
pretax_income                 2     0     0      0      0  -                    -
repurchases                   2     0     0      0      0  -                    -
stock_based_compensation      2     0     0      0      0  -                    -
dividends_paid                1     0     0      1      0  -      

## 3. Patterns: recurring taxonomy versus company quirks

In [4]:
print(format_pattern_view(report))

Top blocking metrics (companies blocked):
  current_debt              1
  operating_income          1
Diagnosis categories (company-metric findings):
  ALTERNATIVE_CONCEPT    1
  COMPOSITE_CANDIDATE    1
  MISSING_CONCEPT        2
  STALE_CONCEPT          2
Silently stale AVAILABLE metrics (selected concept has no recent value): 1 in 1 companies
  LULU   cash
Ambiguous duplicates by restatement kind:
Composite-metric candidates:
  NKE    current_debt     unsupported: needs a sum of LongTermDebtCurrent + ShortTermBorrowings
Alternative concepts (recurring = 3+ companies):
  quirk     current_debt           ShortTermBorrowings                                NKE,LULU
  quirk     current_debt           LongTermDebtCurrent                                NKE


## 4. The most important finding: silent staleness

The SEC selector takes the first preferred concept that has *any* annual history. When a company stops tagging that concept and moves to another, OwnerLens keeps using the old values, marks the metric `AVAILABLE`, and coverage looks clean. NVIDIA's revenue series, for example, ends in FY2022, and Amazon's capital expenditures end in FY2016. Calculations only avoid wrong numbers because fiscal-year alignment leaves the recent years empty.

In [5]:
for company in report.surveyed:
    for metric in company.silently_stale():
        print(f'{company.ticker:<5} {metric:<24} {company.metrics[metric].reason[:120]}')

LULU  cash                     selected CashAndCashEquivalentsAtCarryingValue was last reported FY2019 ([2014, 2015, 2016, 2017, 2018, 2019]); current 


## 5. What would the next normalization improvement unblock?

`unblocked_by` is a what-if over the diagnoses. It counts companies whose every unsupported or invalid required input would be resolved by an improvement. It does not recompute anything, and it ignores silent staleness, so it measures blocked coverage, not trustworthiness.

In [6]:
current_debt = lambda t, f: f.metric == 'current_debt' and f.diagnosis in (D.ALTERNATIVE_CONCEPT, D.COMPOSITE_CANDIDATE, D.MISSING_CONCEPT)
ambiguity = lambda t, f: f.diagnosis is D.AMBIGUOUS_DUPLICATE and f.restatement in (R.PRECISION, R.STOCK_SPLIT)
fallback = lambda t, f: f.diagnosis in (D.ALTERNATIVE_CONCEPT, D.STALE_CONCEPT) and bool(f.candidates)
long_term_debt = lambda t, f: f.metric == 'long_term_debt' and f.diagnosis is not D.PERIOD_ISSUE

scenarios = {
    'A. current-debt policy (fallbacks + composite + absence)': current_debt,
    'B. restatement-aware ambiguity (precision + stock split)': ambiguity,
    'C. recency-aware concept fallback': fallback,
    'A + B': lambda t, f: current_debt(t, f) or ambiguity(t, f),
    'A + B + C': lambda t, f: current_debt(t, f) or ambiguity(t, f) or fallback(t, f),
    'A + B + C + long-term debt': lambda t, f: current_debt(t, f) or ambiguity(t, f) or fallback(t, f) or long_term_debt(t, f),
}
blocked = [c.ticker for c in report.surveyed if c.unusable_inputs()]
print(f'{len(blocked)} of {len(report.companies)} companies have at least one blocking input\n')
for name, resolves in scenarios.items():
    unblocked = report.unblocked_by(resolves)
    print(f'{name:<60} unblocks {len(unblocked):>2}: {", ".join(unblocked)}')

2 of 2 companies have at least one blocking input

A. current-debt policy (fallbacks + composite + absence)     unblocks  0: 
B. restatement-aware ambiguity (precision + stock split)     unblocks  0: 
C. recency-aware concept fallback                            unblocks  0: 
A + B                                                        unblocks  0: 
A + B + C                                                    unblocks  0: 
A + B + C + long-term debt                                   unblocks  1: LULU


## 6. Acceptance answers

In [7]:
total = len(report.companies)
dist = report.distribution()
print('1-2. Coverage:', ', '.join(f'{k} {v}/{total} ({v / total:.0%})' for k, v in dist.items()))
print('3.   Top blocking metrics:', ', '.join(f'{m} ({n})' for m, n in report.top_blockers()))
recurring = [r for r in report.recurring_candidates() if r[3]]
print(f'4.   Recurring alternative concepts (3+ companies): {len(recurring)};',
      'quirks:', len(report.recurring_candidates()) - len(recurring))
composites = report.composites()
print(f'5.   Composite candidates: {len(composites)} in {len({c[0] for c in composites})} companies',
      f'({sum(1 for c in composites if c[2])} value-level understatements)')
stale = [c for c in report.surveyed if c.silently_stale()]
print(f'     Silently stale metrics: {sum(len(c.silently_stale()) for c in stale)} in {len(stale)} companies')
print('     Diagnosis mix:', report.diagnosis_counts())

1-2. Coverage: FULL 0/2 (0%), PARTIAL 1/2 (50%), FAILED 1/2 (50%)
3.   Top blocking metrics: current_debt (1), operating_income (1)
4.   Recurring alternative concepts (3+ companies): 0; quirks: 2
5.   Composite candidates: 1 in 1 companies (0 value-level understatements)
     Silently stale metrics: 1 in 1 companies
     Diagnosis mix: {'ALTERNATIVE_CONCEPT': 1, 'COMPOSITE_CANDIDATE': 1, 'MISSING_CONCEPT': 2, 'STALE_CONCEPT': 2}


**Reading the answers**

* **Coverage:** 3 of 24 companies (12%) reach full coverage, 12 (50%) are partial, and 9 (38%) fail. A failure means owner economics itself is unavailable.
* **Top blockers:** current debt dominates (9 companies blocked, 12 unsupported). Next come diluted shares (5, all stock-split or precision restatements), operating income (4: energy, pharma, Nike), long-term debt (3), and revenue (2).
* **Recurring versus quirk:** failures are **mostly recurring taxonomy patterns**. Three current-debt alternatives each recur in 3–4 companies. Every ambiguous duplicate is a precision re-rounding (8) or a stock-split restatement (4). The two genuine revisions sit inside stale concepts. Staleness recurs across 10 companies.
* **Composite metrics:** current debt is composite in 5 companies (3 unsupported, 2 value-level understatements, including MSFT FY2024 commercial paper). No other metric needed a composite.
* **SEC-first at 100+:** practical, *if* the next improvements are general rules rather than per-ticker overrides. The blockers cluster into a handful of policies, and request volume is two SEC calls per company. Per-ticker overrides would not scale: 21 of 24 companies fall short of `FULL`, and 19 have a blocking input.
* **Next improvement:** see the Feature 6 documentation for the ranked recommendation. Recency-aware concept selection comes first, because it is a *correctness* fix. Current-debt policy and restatement-aware ambiguity handling are the largest coverage gains.